# Assignment 4

# Problem Description

This report decomstrate a method for evaluating numerical eigenvalues of matrices, say $M\in\mathbb{R}^{n\times n}$.

## Algorithm

Adopting the **Power Method**, we can evaluate the eigenvalue of $M$ which is real and takes the largest absolute value. Suppose $\lambda(M)=\{\lambda_i\}_{i\in I}, I=\{1,2,\cdots,n\}$, each occurs algebraic multiplicity times. And WLOG, $|\lambda_i|\geqslant |\lambda_{i+1}|$, and denote the corresponding eigenspace basis (the first is eigenvector in each group) as $\{v_i\}_{i\in I}$, then for any $v\in \mathbb{R}^n$, we have

$$ v=\lambda_1v_1 + \lambda_2 v_2+\cdots+\lambda_n v_n $$

in which for any $v_i$, either $Mv_i=\lambda_i v_i$ or $Mv_i=v_{i-1}$, $\exists k_i\in\mathbb{N}$, such that $M^{k_i}v_i$ is an eigenvector. Hence from some $k>N\in N$, we have

$$ v^{(k)}=M^kv=(\lambda_{m_1}^k+o(\lambda_{m_1}^k))v_{m_1}+(\lambda_{m_2}^k+o(\lambda_{m_2}^k))v_{m_2}+\cdots $$

where $1=m_1>m_2>\cdots$, thus $|\lambda_{m_1}|>|\lambda_{m_2}|>\cdots$, and

$$\lim_{k\to +\infty}\dfrac{v^{(k)}}{|\lambda|^k} =\lim_{k\to +\infty}\dfrac{1}{|\lambda|^k} (\lambda_{m_1}^k+o(\lambda_{m_1}^k))v_{m_1}+(\lambda_{m_2}^k+o(\lambda_{m_2}^k))v_{m_2}+\cdots =\text{sign}\lambda_{1} v_1$$

Also, if we use the infinite norm to normalize $x^{(k)}$ and get $\hat{x}^{(k)}=\dfrac{x^{(k)}}{||x^{(k)}||_{\infty}}$ before next iteration, the fact above can be written as 

$$\lim_{k\to +\infty}\dfrac{M\hat{x}^{(k)}}{\hat{x}^{(k)}}=\lambda_1\quad\text{(item-wise quotient)}$$

Thus we can iterate enough times to get a rough estimate for the eighenvalue of the largest absolute value.

We can also write $\lambda^{(k)}={\hat{x}^{(k)}}^TM\hat{x}^{(k)}/{\hat{x}^{(k)}}^T\hat{x}^{(k)}$ as the estimation for $\lambda$.

A variation for this method is the **Inverse Power Method**, this method is intuitively based on the observation that if $\lambda, v$ are one eigenvalue and the corresponding eigenvector of $M$, and $M$ is invertible and $\lambda\neq 0$, we'd have

$$M v=\lambda v, \dfrac{1}{\lambda} v=M^{-1}v$$

that is, $\dfrac{1}{\lambda}, v$ are one eigenvalue and the corresponding eigenvector of $M^{-1}$. Hence we can evaluate the inverse of $M$ and apply the *Power Method*, we'd get the absolutely largest eigenvalue of $M^{-1}$, i.e the absolutely smallest eigenvalue of $M$, together with its eigenvector.

In real application, since not all $M$'s are invertible, a pertubation usually be added to the iteration formula:

$$x^{(k+1)}=\dfrac{(M-p I)^{-1}x^{(k)}}{||x^{(k)}||_{\infty}}$$

Since $M-pI$ is constant, it's okay to do a decomposition, for example, LU decomposition first.

Python's numpy module provides a built-in function `eigs` to calculate the eigenvalues of a matrix.

Another approach is the so-called **Deflation Method**, based on the idea that if we already know the absolutely largest eigenvalue, then we can construct another matrix out of $M$ that the second largest becomes the largest for the new matrix. Consider $B=A-\lambda_1 x^T$, we can observe that 

$$Bv_1=Av_1-\lambda_1v_1x^Tv_1=\lambda_1v_1-\lambda_1v_1=0$$

but $v_j,j>1$ is not the eigenvectors of $B$ now, so we need to take $w_j$ s.t. $x^T w_j=c+x^Tv_j$, where $c=-\lambda_1x^Tv_j/\lambda_j$. 

- Use the power method to obtain $\lambda_1,v_1$
- Construct deflated matrix $B=A-\lambda_1v_1x^T, x^Tv_1=1$
- Calculate $w_2$ and apply power method on $B$ to obtain $\lambda_2$ and $w_2$ then recover $v_2$
    $$ v_2=(\lambda_2-\lambda_1)w_2+\lambda_1(x^Tw_2)v_1$$

The **Weilandt deflation** takes $x=\dfrac{1}{\lambda_1(v_1)_1}R_1$, where $(v_1)_1$ is the first component of $v_1$ and $R_1$ is the first row of $A$.

# Results

The distribution of elements in the random matrix is plotted below:

![mat](mat.png)
**Figure 1:** Distribution of matrix

**Power method:** The converging process is shown below (errors of vector measured by 2-norm), 

![err1](power_error.png)

**Figure 2:** Power method

![err2](power_error2.png)

**Figure 3:** Inverse power method

and the table

|   | Iteration count | $\lambda$ | $\lambda$ time | $\lambda$ error | vector error |
|:--:|:---------:| :----:|:-------:| :------:| :----------:|
| Largest | 132 | 204.766977 | 0.057198 | $7.389644\times 10^{-13}$ | $2.837341\times 10^{-9}$ |
| Smallest | 941 | 9.000001 | 0.735026 | $6.217248\times 10^{-14}$ | $3.291542\times 10^{-9}$|
| Second Largest (Deflated) | - | 178.602105 | 0.192601  | $1.296980\times 10^{-5}$ | $1.052203\times 10^{-7}$ |
| Second Smallest (Deflated) | - |  9.135542  | 0.934192 | $1.118179\times 10^{-7}$ | $1.067390\times 10^{-5}$ |

**Table 1:** Errors, iteration counts, and times

And more detailed info is left in the appendix. 


## Analysis

From the table, we can observe that the inverse powermethod takes far more iterations and time to converge to the same accuracy, part of the reason is that we toke the inverse of $M$, which is time consuming and may introduces new errors. The results' error are all small enough to accept.

# Collaboration

No collaboration on this project.

# Academic Integrity

On my personal intergrity as a student and member of the UCD community, I'm not given or received any unauthorized assitance on this assignment.

In [134]:
import numpy as np
from matplotlib import pyplot as plt
from ttictoc import tic, toc
from tqdm import tqdm
from tabulate import tabulate
import scipy.linalg as la

In [151]:
def power(A, x0, tol=1e-8, max_iter=1000, collect=False):
    x1 = np.ones_like(x0)
    x1 = x1 * np.inf
    i = 0
    lams = []
    points = [x0]
    while np.linalg.norm(x1 - x0) > tol and i < max_iter:
        if i > 0:
            x0 = x1
        i += 1
        x1 = A @ x0
        x1 /= np.linalg.norm(x1, 2)
        if collect:
            lams.append(np.dot(x1, A @ x1) / np.dot(x1, x1))
            points.append(x1)
    return np.dot(x1, A @ x1) / np.dot(x1, x1), x1, \
            (lams if collect else None), (points if collect else None), i

In [152]:
eps = np.finfo(float).eps

def LUdecomposition(A: np.ndarray) -> np.ndarray:
    n = len(A)
    L = np.identity(n)
    U = A.copy()

    for j in range(0, n - 1):
        for i in range(j + 1, n):
            if abs(U[j][j]) < eps:
                return None # type: ignore # without pivoting, we cannot handle this
            m = U[i][j] / U[j][j]
            U[i, :] = U[i, :] - m * U[j, :]
            L[i, j] = m
    
    assert(np.linalg.norm(L @ U - A, 2) < 0.0001)
    return L, U # type: ignore

def forward_substitution(L: np.ndarray, b: np.ndarray) -> np.ndarray | None:
    x = np.zeros_like(b)

    if abs(L[0, 0]) < eps:
        return None
    
    x[0] = b[0] / L[0, 0]
    for i in range(1, len(b)):
        x[i] = (b[i] - np.dot(x[:i], L[i,:i])) / L[i][i]

    assert(np.linalg.norm(L @ x - b, 2) < 0.0001) # type: ignore
    return x

def back_substitution(U: np.ndarray, b: np.ndarray) -> np.ndarray | None:
    x = np.zeros_like(b)

    if abs(U[-1, -1]) < eps:
        return None
    
    x[-1] = b[-1] / U[-1, -1]
    for i in range(len(b), 0, -1):
        x[i - 1] = (b[i - 1] - np.dot(U[i - 1, i:], x[i:])) / U[i - 1, i - 1]

    assert(np.linalg.norm(U @ x - b, 2) < 0.0001)

    return x

def solveLUb(L: np.ndarray, U: np.ndarray, b: np.ndarray) -> np.ndarray | None:
    y = forward_substitution(L, b)
    if y is None:
        return y
    
    x = back_substitution(U, y)

    return x

In [147]:
def inverse_power(A, x0, mu, tol=1e-8, max_iter=1000, collect=False):
    # Ainv = np.linalg.inv(A - mu * np.identity(len(x0)))
    L, U = LUdecomposition(A - mu * np.identity(len(x0)))
    Ainv = np.zeros_like(A)
    for i in range(len(x0)):
        b = np.zeros_like(x0)
        b[i] = 1
        Ainv[:, i] = solveLUb(L, U, b)
    
    res = power(Ainv, x0, tol, max_iter, collect)
    res = list(res)
    res[0] = 1 / res[0] # type: ignore
    if res[2] is not None:
        res[2] = [1 / m for m in res[2]]  # type: ignore
    return tuple(res)

In [117]:
def second_power(A, x0, tol=1e-8, max_iter=1000):
    lam1, v1, _, _, _ = power(A, x0, tol, max_iter)
    x = A[0, :] / (lam1 * v1[0]) # Weilandt defaltion
    B = A - lam1 * np.array([[a * b for b in v1] for a in x])
    lam2, w2, _, _, _ = power(B, x0, tol, max_iter)
    v2 = (lam2 - lam1) * w2 + lam1 * np.array([[a * b for b in w2] for a in x]) @ v1
    return lam2, v2


In [148]:
def inverse_second_power(A, x0, mu, tol=1e-8, max_iter=1000):
    L, U = LUdecomposition(A - mu * np.identity(len(x0)))
    Ainv = np.zeros_like(A)
    for i in range(len(x0)):
        b = np.zeros_like(x0)
        b[i] = 1
        Ainv[:, i] = solveLUb(L, U, b)
    res = second_power(Ainv, x0, tol, max_iter)
    res = list(res)
    res[0] = 1 / res[0] # type: ignore
    return res

In [ ]:
n = 50
B = np.random.normal(0, 1, (n, n))
A = 10 * np.identity(n) + B + B.T + B.T @ B

fig = plt.figure(figsize=(10, 5))
ax = plt.subplot(1, 2, 1)

plt.imshow(A)
plt.colorbar()

ax = plt.subplot(1, 2, 2)

plt.hist(A.flatten(), 30)
plt.title("Random matrix A's element distribution")
plt.xlabel("Value")
plt.ylabel("Count")
plt.show()

fig.savefig("mat.png")

In [120]:
def get_sorted(A):
    sys = np.linalg.eig(A)
    raw = [(val, sys[1][:, i] / np.linalg.norm(sys[1][:, i])) for i, val in enumerate(sys[0])]
    return sorted(raw, key=lambda x: x[0], reverse=True)

In [153]:
res = power(A, np.ones(n), tol=1e-8, collect=True)
eig = get_sorted(A)

tmp = res[1] / np.linalg.norm(res[1], 2)
if eig[0][1][0] / tmp[0] < -0.5:
    tmp = -tmp
print(f"Iteration: {res[-1]}")
print(f"Eigenvalue: {res[0]} \n Ref: {eig[0][0]} \n Error: {np.abs(res[0] - eig[0][0])}")
print(f"Eigenvector: {tmp} \n Ref: {eig[0][1]} \n Error: {np.linalg.norm((A @ tmp - res[0] * tmp), np.inf) / np.abs(eig[0][0])}")


Iteration: 132
Eigenvalue: 204.76697719973834 
 Ref: 204.76697719973907 
 Error: 7.389644451905042e-13
Eigenvector: [ 0.01345037  0.10050282  0.02419643  0.32636274 -0.23514337 -0.14946472
 -0.09540664  0.20612816  0.04984585  0.19853698  0.13989715  0.22178551
 -0.05405177 -0.04862444 -0.00716654  0.03717647 -0.04867027  0.06126735
  0.01016591  0.36648351 -0.16567189  0.20966157  0.00520087  0.0756438
 -0.07984955  0.07144564 -0.02863688 -0.02406511  0.09833262 -0.07164729
 -0.07786056 -0.03055941 -0.11604063 -0.0869426  -0.05884986 -0.06389616
 -0.06545324 -0.01645593 -0.15599986 -0.14292567 -0.04202866 -0.06808073
 -0.08582559  0.17643141 -0.17094074 -0.09926419  0.00477796 -0.45666353
 -0.09226072  0.06864513] 
 Ref: [ 0.01345036  0.10050281  0.02419643  0.32636275 -0.23514335 -0.14946472
 -0.09540663  0.20612817  0.04984584  0.19853698  0.13989714  0.22178551
 -0.05405176 -0.04862443 -0.00716655  0.03717645 -0.04867028  0.06126734
  0.01016592  0.36648352 -0.16567187  0.20966159 

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = plt.subplot(1, 2, 1)

plt.plot(res[2])
plt.title("Convergence of eigenvalue")
plt.xlabel("Iteration")

ax = plt.subplot(1, 2, 2)

plt.plot(np.abs(res[2] - eig[0][0]))
plt.title("Convergence of eigenvalue error")
plt.yscale('log')
plt.xlabel("Iteration")
plt.show()

fig.savefig("power_error.png")

In [154]:
res = inverse_power(A, np.ones(n), 0, tol=1e-8, max_iter=10000, collect=True)

tmp = res[1] / np.linalg.norm(res[1], 2) # type: ignore
if eig[-1][1][0] / tmp[0] < -0.5:
    tmp = -tmp

print(f"Iteration: {res[-1]}")
print(f"Eigenvalue: {res[0]} \n Ref: {eig[-1][0]} \n Error: {np.abs(res[0] - eig[-1][0])}")
print(f"Eigenvector: {tmp} \n Ref: {eig[-1][1]} \n Error: {np.linalg.norm((A @ tmp - res[0] * tmp), np.inf) / np.abs(eig[-1][0])}")

Iteration: 941
Eigenvalue: 9.000001376381688 
 Ref: 9.000001376381626 
 Error: 6.217248937900877e-14
Eigenvector: [ 0.04368178  0.02381352  0.04198489 -0.15699449 -0.15443373 -0.38905424
  0.04431851 -0.04622965 -0.12592108  0.14822254  0.30677425 -0.01926446
 -0.0763792  -0.18094966  0.25472214 -0.16510265 -0.03384475 -0.13659364
  0.13086509  0.0843065  -0.00841302 -0.17568843  0.10812219 -0.17342886
  0.06004379  0.07655097 -0.13407885  0.29126472 -0.17854401  0.18995586
  0.03748892 -0.1118446   0.01994137 -0.00443125  0.1335485  -0.01120299
  0.07963393 -0.00550082  0.00226122 -0.13660182 -0.16276126 -0.22683251
 -0.08377267  0.07682681 -0.02579743 -0.01639528 -0.13415639  0.19415655
 -0.02036255 -0.1629598 ] 
 Ref: [ 0.04368174  0.02381374  0.04198488 -0.1569945  -0.1544336  -0.38905414
  0.04431847 -0.04622963 -0.12592117  0.14822261  0.30677433 -0.01926445
 -0.07637924 -0.18094981  0.25472201 -0.1651027  -0.03384459 -0.13659349
  0.13086519  0.08430659 -0.00841311 -0.17568848  

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = plt.subplot(1, 2, 1)

plt.plot(res[2])
plt.title("Convergence of eigenvalue")
plt.xlabel("Iteration")

ax = plt.subplot(1, 2, 2)

plt.plot(np.abs(res[2] - eig[-1][0]))
plt.title("Convergence of eigenvalue error")
plt.yscale('log')
plt.xlabel("Iteration")
plt.show()

fig.savefig("power_error2.png")

In [155]:
res = second_power(A, np.ones(n), max_iter=10000)

tmp = res[1] / np.linalg.norm(res[1], 2)
if eig[1][1][0] / tmp[0] < -0.5:
    tmp = -tmp

print(f"Eigenvalue: {res[0]} \n Ref: {eig[1][0]} \n Error: {np.abs(res[0] - eig[1][0])}")
print(f"Eigenvector: {tmp} \n Ref: {eig[1][1]} \n Error: {np.linalg.norm((A @ tmp - res[0] * tmp), np.inf) / np.abs(eig[1][0])}")

Eigenvalue: 178.60211232505802 
 Ref: 178.6020993552498 
 Error: 1.2969808210527844e-05
Eigenvector: [ 0.10165128  0.07972828  0.05709685 -0.12968956 -0.29384795  0.03230216
 -0.10508773 -0.09530698  0.07169587  0.00248732  0.12729387 -0.13070948
 -0.06212848 -0.11157831  0.04791088  0.22349627  0.07982841  0.12004066
 -0.10691273 -0.12048087 -0.24428523 -0.24544245 -0.11183434 -0.04720798
 -0.06585548 -0.33883483 -0.04280289 -0.06214395  0.05522506  0.19170813
 -0.09085172 -0.15170034 -0.13294638  0.10902843 -0.14165458  0.01822115
 -0.03962231  0.0075498  -0.32324607  0.28109783 -0.09845714 -0.04876725
 -0.07619542 -0.15669221  0.0184524   0.22504308  0.17711171 -0.09861606
 -0.04203899  0.00814475] 
 Ref: [ 0.10164909  0.07972817  0.0570965  -0.12968964 -0.2938477   0.03230229
 -0.10508756 -0.09530721  0.07169572  0.00248757  0.12729366 -0.13070961
 -0.06212818 -0.11157862  0.04791075  0.22349637  0.07982864  0.12004041
 -0.10691271 -0.12048136 -0.24428533 -0.24544219 -0.11183469 -0

In [156]:
res = inverse_second_power(A, np.ones(n), 0, max_iter=10000)

tmp = res[1] / np.linalg.norm(res[1], 2)
if eig[-2][1][0] / tmp[0] < -0.5:
    tmp = -tmp

print(f"Eigenvalue: {res[0]} \n Ref: {eig[-2][0]} \n Error: {np.abs(res[0] - eig[-2][0])}")
print(f"Eigenvector: {tmp} \n Ref: {eig[-2][1]} \n Error: {np.linalg.norm((A @ tmp - res[0] * tmp), np.inf) / np.abs(eig[-2][0])}")

Eigenvalue: 9.135542279547566 
 Ref: 9.135542391365517 
 Error: 1.1181795045445142e-07
Eigenvector: [-0.05382322  0.33132369 -0.0091612  -0.01139003  0.19195163  0.14646111
 -0.06344929  0.03622346 -0.14421107  0.09478826  0.10877418  0.02541934
 -0.05575339 -0.23457236 -0.19294146 -0.07291432  0.23902503  0.23953944
  0.14979466  0.13455301 -0.13023228 -0.06815443 -0.02741016 -0.2937932
 -0.18088811  0.008706   -0.07530836 -0.03856669 -0.1896224  -0.07234456
  0.15688669 -0.08938507 -0.12984435  0.05761935 -0.01620196 -0.23693752
  0.0349701  -0.05973417  0.10637106  0.09967903  0.02153749  0.22228422
  0.12252432  0.24141868 -0.02696518 -0.05322631  0.16865599  0.15952975
  0.10108279  0.09800391] 
 Ref: [-0.05381832  0.33132307 -0.00916244 -0.0113893   0.19195224  0.14646166
 -0.06344911  0.03622244 -0.14421179  0.09478867  0.10877366  0.02541971
 -0.05575339 -0.23457253 -0.19294181 -0.07291418  0.23902481  0.23953939
  0.14979485  0.13455169 -0.13023336 -0.0681536  -0.02741036 -0.2

In [149]:
m = 25

tic()
for _ in range(m): power(A, np.ones(n), max_iter=10000)
print(f"Time: {toc()}")

tic()
for _ in range(m): inverse_power(A, np.ones(n), 0, max_iter=10000)
print(f"Time (inverse): {toc()}")

tic()
for _ in range(m): second_power(A, np.ones(n), max_iter=10000)
print(f"Time (deflated): {toc()}")

tic()
for _ in range(m): inverse_second_power(A, np.ones(n), 0, max_iter=10000)
print(f"Time (inverse deflated): {toc()}")

Time: 0.05719790002331138
Time (inverse): 0.7350258999504149
Time (deflated): 0.1926007999572903
Time (inverse deflated): 0.9341918000718579
